In [3]:
# stl_lstm_forecast_app.py

import streamlit as st
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.seasonal import STL

# ------------------------------------------
# ✅ UI: Sidebar Inputs
# ------------------------------------------
st.title("📈 Stock Price Forecast using STL + LSTM")
st.sidebar.header("Settings")

ticker = st.sidebar.text_input("Enter Ticker Symbol", "HDFCBANK.NS")
start_date = st.sidebar.date_input("Start Date", datetime(2020, 1, 1))
end_date = st.sidebar.date_input("End Date", datetime(2025, 8, 6))
window_size = st.sidebar.slider("Window Size (LSTM Input)", min_value=5, max_value=50, value=10, step=1)
epochs = st.sidebar.slider("Training Epochs", min_value=5, max_value=100, value=10, step=5)
st.sidebar.markdown("---")

if st.sidebar.button("Run Forecast"):
    # ------------------------------------------
    # ✅ 1. Load Data
    # ------------------------------------------
    st.subheader(f"📊 Loading Data for: {ticker}")
    data = yf.download(ticker, start=start_date, end=end_date)

    if data.empty:
        st.error("No data retrieved. Please check the ticker or date range.")
    else:
        d_high = data["High"]
        st.write("Last 5 entries:")
        st.dataframe(d_high.tail())

        # ------------------------------------------
        # ✅ 2. STL Decomposition
        # ------------------------------------------
        st.subheader("🔍 STL Decomposition")
        st.line_chart(d_high)

        stl = STL(d_high, period=30)
        result = stl.fit()
        trend = result.trend
        seasonal = result.seasonal

        st.line_chart(pd.DataFrame({"Trend": trend, "Seasonal": seasonal}))

        # ------------------------------------------
        # ✅ 3. Prepare Data for LSTM
        # ------------------------------------------
        def prepare_lstm_data(series, window_size):
            scaler = MinMaxScaler()
            scaled = scaler.fit_transform(series.values.reshape(-1, 1))
            X, y = [], []
            for i in range(len(scaled) - window_size):
                X.append(scaled[i:i+window_size])
                y.append(scaled[i+window_size])
            return np.array(X), np.array(y), scaler

        X_trend, y_trend, scaler_trend = prepare_lstm_data(trend, window_size)
        X_seasonal, y_seasonal, scaler_seasonal = prepare_lstm_data(seasonal, window_size)

        # ------------------------------------------
        # ✅ 4. Train LSTM Models
        # ------------------------------------------
        @st.cache_resource(show_spinner=True)
        def build_and_train_lstm(X, y, epochs):
            model = Sequential([
                LSTM(5, return_sequences=True, input_shape=(X.shape[1], 1)),
                LSTM(5, return_sequences=True),
                LSTM(5, return_sequences=True),
                LSTM(5, return_sequences=False),
                Dense(1)
            ])
            model.compile(optimizer='adam', loss='mse')
            model.fit(X, y, epochs=epochs, batch_size=10, verbose=0)
            return model

        st.subheader("🤖 Training LSTM Models")
        model_trend = build_and_train_lstm(X_trend, y_trend, epochs)
        model_seasonal = build_and_train_lstm(X_seasonal, y_seasonal, epochs)

        # ------------------------------------------
        # ✅ 5. Predict & Inverse Transform
        # ------------------------------------------
        y_trend_pred = model_trend.predict(X_trend)
        y_seasonal_pred = model_seasonal.predict(X_seasonal)

        trend_pred = scaler_trend.inverse_transform(y_trend_pred)
        seasonal_pred = scaler_seasonal.inverse_transform(y_seasonal_pred)

        final_pred = trend_pred.flatten() + seasonal_pred.flatten()
        actual = d_high.values[window_size:]

        # ------------------------------------------
        # ✅ 6. Plot Predictions
        # ------------------------------------------
        st.subheader("📈 Actual vs Forecasted")
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(actual, label="Actual", linewidth=2)
        ax.plot(final_pred, label="Predicted (Trend + Seasonal)", linestyle='--')
        ax.set_title("LSTM Forecast on STL Components")
        ax.legend()
        ax.grid(True)
        st.pyplot(fig)

        # ------------------------------------------
        # ✅ 7. RMSE
        # ------------------------------------------
        rmse = np.sqrt(mean_squared_error(actual, final_pred))
        st.success(f"📉 Final RMSE: {rmse:.4f}")

        # ------------------------------------------
        # ✅ 8. Forecast Next Day
        # ------------------------------------------
        last_trend_window = trend.values[-window_size:].reshape(1, window_size, 1)
        last_trend_scaled = scaler_trend.transform(last_trend_window.reshape(window_size, 1)).reshape(1, window_size, 1)
        next_trend_scaled = model_trend.predict(last_trend_scaled)
        next_trend = scaler_trend.inverse_transform(next_trend_scaled)[0][0]

        last_seasonal_window = seasonal.values[-window_size:].reshape(1, window_size, 1)
        last_seasonal_scaled = scaler_seasonal.transform(last_seasonal_window.reshape(window_size, 1)).reshape(1, window_size, 1)
        next_seasonal_scaled = model_seasonal.predict(last_seasonal_scaled)
        next_seasonal = scaler_seasonal.inverse_transform(next_seasonal_scaled)[0][0]

        next_day_forecast = next_trend + next_seasonal
        st.info(f"📅 Forecast for Next Day High: `{next_day_forecast:.2f}`")

2025-10-16 04:46:28.252 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-16 04:46:28.613 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2025-10-16 04:46:28.614 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-16 04:46:28.615 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-16 04:46:28.615 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-16 04:46:28.616 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-16 04:46:28.617 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-16 04:46:28.618 Thread 'MainThread': mi

In [2]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 129.6 MB/s eta 0:00:00
